# orchestration workers: practice

Configure a real SDK agent and inspect its tool schema → Client, request, and response are separate objects → An agent loop must make progress and stop → Await vs. concurrent work → A guardrail makes a stop/continue decision → Plan → search → synthesize → A callback receives a message and history → A recurring engine is different from a UI

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [2_openai/community_contributions/workflow_patterns/4-Orchestration_Workers/orchestration_workers.ipynb](../../../../../../../2_openai/community_contributions/workflow_patterns/4-Orchestration_Workers/orchestration_workers.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configure a real SDK agent and inspect its tool schema

Agent holds instructions/model/tools. The function_tool decorator constructs a schema from the signature and docstring. Constructing this configuration does not run the agent or contact a provider.

**Predict:** Which argument names will appear in the registered tool schema?

In [ ]:
from agents import Agent, function_tool, set_tracing_disabled
set_tracing_disabled(True)
@function_tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b
agent = Agent(name="Practice teacher", instructions="Use add_numbers for addition.",
              model="gpt-4o-mini", tools=[add_numbers])
print(agent.name, agent.tools[0].name)
print(sorted(agent.tools[0].params_json_schema["properties"]))
assert sorted(agent.tools[0].params_json_schema["properties"]) == ["a", "b"]

**Check:** a and b. The schema advertises the function; Runner executes the model/tool workflow when deliberately invoked.

**Change one thing:** Change the docstring and inspect the tool description. Find Runner.run in the source: that is where the interaction loop actually begins.

## 2. Client, request, and response are separate objects

Constructing a client prepares a connection interface. A request invokes it. The returned response contains choices, a message, and its content. This fixture imitates that nested shape.

**Predict:** What does choices[0] select?

In [ ]:
from types import SimpleNamespace
response = SimpleNamespace(choices=[
    SimpleNamespace(message=SimpleNamespace(content="The answer is 4."))
])
answer = response.choices[0].message.content
print(answer)
assert answer == "The answer is 4."

**Check:** The first returned choice. This is simulated response data, not a model call.

**Change one thing:** Add a second choice with different text. Read choices[1] and explain why choosing an index matters.

## 3. An agent loop must make progress and stop

A loop repeats model/tool steps until a final answer or a configured limit. This fixed script of decisions shows the order without executing generated code or calling a model.

**Predict:** Which decision ends the loop?

In [ ]:
decisions = iter([
    {"kind": "tool", "name": "add", "a": 12, "b": 30},
    {"kind": "answer", "text": "The sum is 42."},
])
events = []
for turn in range(3):  # Bound the number of steps.
    decision = next(decisions)
    if decision["kind"] == "tool":
        if decision["name"] != "add":
            raise ValueError("Unknown tool")
        events.append({"tool_result": decision["a"] + decision["b"]})
    else:
        events.append({"final": decision["text"]})
        break
else:
    raise RuntimeError("No final answer within the limit")
print(events)
assert events[0]["tool_result"] == 42

**Check:** The answer decision ends it. A loop that only keeps requesting tools needs a bound and an explicit failure path.

**Change one thing:** Add another tool step before the answer. Keep a finite turn limit and explain how the stopping rule changes.

## 4. Await vs. concurrent work

Calling an async function creates a coroutine. Await executes/waits for it. gather schedules independent coroutines and returns results in input order. Notebook cells support top-level await.

**Predict:** Will results follow completion order or input order?

In [ ]:
import asyncio
async def work(label, delay):
    await asyncio.sleep(delay)  # Yield control while waiting.
    return label
results = await asyncio.gather(work("A", 0.002), work("B", 0))
print(results)
assert results == ["A", "B"]

**Check:** Input order. In a standalone script, wrap work in async def main and call asyncio.run(main()); do not do that inside a running notebook loop.

**Change one thing:** Run the two functions with separate awaits. Explain why a dependent stage cannot start until it has its input.

## 5. A guardrail makes a stop/continue decision

An input guardrail checks a request before later work. The tripwire is a boolean. This deterministic example teaches the branch; it is not the model-based name detector in the source.

**Predict:** Does Alice trigger this demonstration guardrail?

In [ ]:
from pydantic import BaseModel
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str
def check_name(message):
    found = "Alice" in message  # Deliberately limited teaching rule.
    return NameCheckOutput(is_name_in_message=found, name="Alice" if found else "")
for request in ["Send from Alice", "Send from Head of Business Development"]:
    result = check_name(request)
    tripwire_triggered = result.is_name_in_message
    print("stop" if tripwire_triggered else "continue")
assert check_name("Send from Alice").is_name_in_message

**Check:** Yes for Alice, no for the title. A boolean decision does not establish that every personal name is detected.

**Change one thing:** Try another name and observe the limitation. Describe how you would evaluate false positives and missed names.

## 6. Plan → search → synthesize

The research workflow separates a search plan, collected findings, and a report. Use fixed fictional snippets first so you can inspect how evidence reaches the writer. No current facts are being researched here.

**Predict:** Will the report receive the original query as well as the findings?

In [ ]:
from dataclasses import dataclass
@dataclass
class SearchItem:
    reason: str
    query: str
query = "How should a beginner practice tool calls?"
plan = [SearchItem("Understand arguments", "tool argument schema"),
        SearchItem("Understand execution", "tool dispatcher")]
fixture = {"tool argument schema": "A schema describes arguments.",
           "tool dispatcher": "Application code executes an allowed function."}
findings = [fixture[item.query] for item in plan]
report = {"query": query, "summary": " ".join(findings), "follow_up_questions": ["How do we test invalid arguments?"]}
print(report)
assert len(findings) == 2 and "Application" in report["summary"]

**Check:** Yes. Plans choose questions, search collects material, and synthesis uses it. Delivery is a separate side effect.

**Change one thing:** Add a third search plan item and a matching fixture. Explain how the original query keeps synthesis focused.

## 7. A callback receives a message and history

A UI invokes your function and displays its return value. Earlier turns are an input, not automatic model memory. This practice calls the callback directly without starting a server.

**Predict:** What changes when history contains two old messages?

In [ ]:
def chat(message, history):
    return f"You asked: {message}. Earlier messages: {len(history)}."
first = chat("Hello", [])
second = chat("Explain more", [{"role":"user", "content":"Hello"}, {"role":"assistant", "content":first}])
print(first, second, sep="\n")
assert "Earlier messages: 2" in second

**Check:** The count becomes two. Gradio wiring is gr.ChatInterface(chat, type="messages"); launch starts the server.

**Change one thing:** Make the callback return an uppercase version of the question. Compare returning a value with printing it.

## 8. A recurring engine is different from a UI

The UI displays state. An engine repeatedly invokes workers. A finite practice loop makes the scheduling visible without leaving a background job running.

**Predict:** How many worker invocations happen with two rounds and four traders?

In [ ]:
import asyncio
calls = []
async def trader_run(name):
    calls.append(name)
    await asyncio.sleep(0)
    return f"{name}: dry run"
for round_number in range(2):
    results = await asyncio.gather(*(trader_run(name) for name in ["A", "B", "C", "D"]))
    print(round_number + 1, results)
assert len(calls) == 8

**Check:** Eight. Real recurring runs can accumulate API usage and change stored account state.

**Change one thing:** Add a market-open boolean that skips a round. Explain why the source while True needs an intentional stop method.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.